<img src="banner-5-coding-with-ai.png" width="100%">
<br>

# Tokens and Per-Interaction Token Limits

What is a token? A token is a partial word, word or a punctuation mark used to ingest messages by an LLM.
For example, this sentence has 6 tokens: "I like to eat apples."

How can we check the number of tokens in a sentence?
This is where tiktoken comes in handy.

[tiktoken](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb) is a fast open-source tokenizer by OpenAI.

Given a text string (e.g., "tiktoken is great!") and an encoding (e.g., "cl100k_base"), a tokenizer can split the text string into a list of tokens (e.g., ["t", "ik", "token", " is", " great", "!"]).

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
encoded = encoding.encode("I like to eat apples.")
print(encoded)

[encoding.decode_single_token_bytes(token) for token in encoded]

In [ ]:
def num_tokens_from_string(string: str, model_name: str="gpt-3.5-turbo") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(model_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string("I like to eat apples.")

The token limit (now known as 'context window limit') for gpt-3.5-turbo is 16,385 tokens (as of February 2025). This limit includes the token count from both the prompt and completion. The number of tokens in the conversation to date, combined with the value of the max_tokens parameter, must stay under the model's context length limit, or you'll receive an error.

See the [OpenAI Cookbook on Tokens](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb) for more details.

*Clarification*: There is no separate “per-interaction” token limit. The only limit is the model’s context window. This window counts everything together:

	system + user + assistant messages (all prompts/responses so far), and
	the new completion tokens.

If the sum exceeds the context window (e.g. 16,385 tokens for gpt-3.5-turbo), the request fails with a context length error.

# Chunking Requests

It stands to reason that given this limit, how can we process larger prompts or files or responses? The answer is chunking. Chunking is the process of breaking down a longer user input (or "prompt") into smaller, more manageable pieces.

**Historical context**: Early GPT models had very small context windows (2k–4k tokens). To process larger inputs, people had to split (“chunk”) text and feed it piece by piece — this was “chunking for interaction.”

**Today**: With much larger windows (128k+), you usually **don’t need to chunk** just to interact. Instead, chunking is mostly about retrieval and efficiency (RAG) rather than model comprehension.
  * Most day-to-day inputs (codebases, PDFs, transcripts) now fit directly.
  * That means you can skip chunking for interaction in many cases.
  * The model doesn’t magically do “better” if you chunk unnecessarily — in fact, chunking can break coherence.

**When chunking is still useful**

Even with a giant context, chunking isn’t dead — but the use case shifts:

* RAG / embeddings:
Chunking lets you embed smaller pieces, retrieve only what’s relevant, and feed that into the model. This saves tokens and improves accuracy.
(If you always dump the entire 128k input, you risk diluting focus and burning money.)

* Extreme scale data:
Whole knowledge bases, multi-hour transcripts, or gigabytes of logs — even 128k won’t cut it. Chunk → embed → retrieve is the only way.

* Performance & cost:
Smaller prompts run faster and cost less. Chunk-and-retrieve helps you avoid bloated prompts.

**Rule of thumb now:**  If everything fits in the context window → no need to chunk.
If it doesn’t, or if you’re building retrieval/knowledge workflows → chunk for RAG, not for interaction.




Chunking can be helpful for a few reasons:

1. **Limited Token Capacity:** LLMs process text in units called tokens. There is a maximum number of tokens they can handle in a single conversation (for GPT-3.5-turbo, this limit is 16,385 tokens), which encompasses both the user's input and the model's output combined. If your input is larger than the context window, chunking is required.

3. **Improving Response Quality:** By addressing parts of input separately, it can sometimes lead to clearer and more precise answers when performing retrieval using chunks. This can be especially helpful if a user has multiple questions or if their prompt contains various topics.

In [ ]:
import tiktoken as tkn
from typing import List

def chunk_prompt(prompt: str, chunk_size: int = 1500, overlap: int = 50) -> List[str]:
    """
    Splits a prompt into chunks of approximately `chunk_size` tokens, with a given overlap.

    Parameters:
    - prompt (str): The text to be chunked.
    - chunk_size (int): The desired number of tokens for each chunk.
    - overlap (int): The number of tokens for overlap between chunks.

    Returns:
    - List[str]: A list of prompt chunks.
    """

    encoding = tkn.encoding_for_model("gpt-3.5-turbo")
    tokens = list(encoding.encode(prompt))

    if len(tokens) <= chunk_size:
        return [prompt]

    chunks = []
    position = 0

    while position < len(tokens):
        start_pos = max(0, position - overlap)
        end_pos = min(position + chunk_size, len(tokens))

        chunk_tokens = tokens[start_pos:end_pos]
        chunk_text = ''.join(encoding.decode_bytes(chunk_tokens).decode('utf-8', errors='ignore'))

        chunks.append(chunk_text)

        position += chunk_size

    return chunks

This function works as follows:

1. Tokenize the prompt using `tiktoken`.
2. If the total tokens are less than or equal to the chunk size, return the whole prompt.
3. Otherwise, proceed to create chunks. Each chunk starts `overlap` tokens before the end of the previous chunk and continues until it has approximately `chunk_size` tokens.

You can adjust `chunk_size` and `overlap` as needed.


## Limitations of Chunking

There are several limitations and challenges associated with the chunking method:

1. **Loss of Context:** Breaking down text can cause loss of context, especially for complex topics.

2. **Increased Complexity:** Managing the sequential processing of chunks adds complexity to the interaction.

3. **Potential Repetition:** The model might provide repetitive responses if overlapping content is too similar.

4. **Token Limitations:** Even with chunking, you're still bound by the token limitations for each conversation.

5. **Response Inconsistency:** Since the model doesn't have a persistent memory of past interactions, providing context manually might lead to variations in responses.

# Conversation Pruning

Pruning conversations is an essential practice when working with Language Models (LLMs). It involves selectively removing or trimming portions of the conversation history to ensure the most relevant and contextually appropriate information is retained. This practice is crucial for several reasons:

1. **Token Limit Management:** LLMs have a maximum token limit for each conversation. Pruning helps maintain the conversation within this limit.

2. **Contextual Relevance:** As conversations progress, earlier messages might become less relevant. Pruning prevents outdated context from influencing the generated response.

3. **Resource Efficiency:** Pruning reduces the computational overhead, making interactions with LLMs more efficient.

4. **User Experience:** Users receive concise and contextually relevant answers without excessive context.

5. **Avoiding Misinterpretation:** Pruning helps focus the context on the current topic, reducing the chance of misinterpretation.

Context: Pruning (or sliding windows) is another historical workaround for small context windows. With bigger windows, you may not need pruning in everyday use — but it’s still helpful when conversations grow very long or when you want to reduce cost and latency by trimming older history.  For software engineering, as we interact with the model over many turns, we do want to prune older messages to keep the conversation within the token limit and maintain relevance.  Some tools like Cline have facilities to help with this.

### Pruning Methods

There are several ways to prune conversations. The most common methods are:

- **Truncation:** Remove the oldest messages in the conversation, keeping only the most recent ones.

- **Selection:** Selectively remove messages based on their relevance to the current topic.

- **Combination:** Combine truncation and selection to remove irrelevant messages and keep the most recent ones.

- **Rephrasing:** Rephrase messages to make them more relevant to the current topic.

- **Reordering:** Reorder messages to ensure the most relevant ones are at the top of the conversation history.

In [ ]:
# Example of a simple pruning function that keeps the first few and last few messages
def prune_context(messages, max_tokens, num_retained=2):
    total_tokens = 0
    pruned_messages = []
    retained_early_messages = messages[:num_retained]
    retained_later_messages = messages[-num_retained:]
    retained_tokens = sum(num_tokens_from_string(message["content"]) for message in retained_early_messages + retained_later_messages)

    for message in retained_early_messages:
        pruned_messages.append(message)
        total_tokens += num_tokens_from_string(message["content"])

    for message in messages[num_retained:len(messages) - num_retained]:
        message_tokens = num_tokens_from_string(message["content"])

        if total_tokens + message_tokens + retained_tokens <= max_tokens:
            pruned_messages.append(message)
            total_tokens += message_tokens
        else:
            break

    for message in retained_later_messages:
        pruned_messages.append(message)

    return pruned_messages

# Summary

* The only limit is the context window, which includes all past prompts and the model’s reply.

* Chunking for interaction was needed when context windows were tiny; now it’s rarely necessary if your input fits.

* Chunking for retrieval (RAG) and pruning for efficiency remain important techniques for working with very large data.

# Conclusion

In this notebook you've:
* Been introduced to the tiktoken library, and how to use it to count tokens in a string.
* Learned about chunking, and how it can be used along with its limitations.
* Learned about conversation pruning, and how it can be used to improve the quality of responses from the AI.
* Learned about tokens, and how they are relevant for chunking and pruning conversations.